## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## specify args

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## Functions

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        # fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))
        fits.append(fit)

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims


def save(fig, fname, dpi=800, do_save=False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "pre-egu-updates")

    ## get fname
    fname = f"{fname}-{varnames[0]}-subtract_median_{REMOVE_MEDIAN}.pdf"

    if do_save:
        fig.savefig(save_dir / fname, dpi=dpi, format="pdf")

    return


def get_perturbed_NRO(params, nro_form_idx=None, nro_type="NROT_Lac"):
    """
    Fix values of R parameter set.
    if 'fix_others' is True, then other parameters are fixed to their
    initial value. Otherwise, given parameter is fixed to its initial value
    """

    ## initialize empty array to hold parameters
    params_new = copy.deepcopy(params)
    params_new[nro_type] = params_new[nro_type].transpose("year", "nro_form", ...)

    ## get numpy version of linear operator
    NRO_Lac = params_new[nro_type].values

    ## update parameter
    if nro_form_idx is None:
        NRO_Lac = NRO_Lac[:1]

    else:
        NRO_Lac[:, nro_form_idx] = NRO_Lac[:1, nro_form_idx]

    ## add back to parameters
    params_new[nro_type] = xr.ones_like(params_new[nro_type]) * NRO_Lac

    return params_new


def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})

    ## differentiate
    ddt_data = data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

    ## mult. by 12 to convert from 1/mo to 1/yr
    return 12 * ddt_data

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)
Th = Th.fillna(0)

#### Load ELI

In [ ]:
## Load ELI
eli_all = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli = src.utils.separate_forced(eli_all)

## add to data
Th = xr.merge([Th, eli])

Scale it

#### Merged Niño index

In [ ]:
## function to get merged nino3/34 regions
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])


## spatial data
forced, anom = src.utils.load_consolidated()

## compute
Th["T_m"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_nino_merged)[
    "sst"
]
Th["T_e"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Te)["sst"]
Th["T_w"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Tw)["sst"]

#### OHC

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


forced_wide, anom_wide = load_consolidated_wide()

In [ ]:
lon_avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean("longitude")

## compute ohc
Th["h_w_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 180)),
)["T"]

## compute ohc
Th["h_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 280)),
)["T"]


## compute ml temp
Th["T_3_ml"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.sel(z_t=slice(None, 50)).integrate("z_t"), (210, 270)),
)["T"]

### NHF

In [ ]:
## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]
nhf_e = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_Te)["nhf"]
T_m = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], fn=get_nino_merged)[
    "sst"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)
Q_e = nhf_e * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_34"] = Q_34.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)
Th["Q_e"] = Q_e.sel(time=Th.time)
Th["Q_m"] = Q_m.sel(time=Th.time)

#### LOAD UPDATED ELI

In [ ]:
eli0 = xr.open_dataarray(DATA_FP / "cesm" / "eli_updated.nc")

Th = xr.merge([Th, eli0])

#### zonal gradient

In [ ]:
def get_ddx(x):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    lon_range_w = dict(longitude=slice(120, 160))
    lon_range_e = dict(longitude=slice(210, 270))

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    return avg(x.sel(**lon_range_w, **lat_range)) - avg(
        x.sel(**lon_range_e, **lat_range)
    )

In [ ]:
## compute dTdx
Th["dTdx_total"] = src.utils.reconstruct_wrapper(
    forced[["sst", "sst_comp"]],
    get_ddx,
)["sst"]

#### load 3-D data

In [ ]:
import pandas as pd

Th_3d = xr.open_mfdataset(
    sorted(list(pathlib.Path(DATA_FP / "cesm" / "Th_3D").glob("*nc"))),
    combine="nested",
    concat_dim="member",
)

## update coordinates to match
Th_3d = Th_3d.assign_coords({"member": np.arange(100)}).compute()
Th_3d = Th_3d.assign_coords({"time": Th.time})
Th_3d = Th_3d.rename({"h": "thermocline"})

In [ ]:
MLD = 50
avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean(["longitude"])
Th_3d["T_3"] = avg(Th_3d[f"T_{MLD}"], (210, 270))
Th_3d["T_34"] = avg(Th_3d[f"T_{MLD}"], (190, 240))
Th_3d["T_4"] = avg(Th_3d[f"T_{MLD}"], (160, 210))
Th_3d["T_e"] = avg(Th_3d[f"T_{MLD}"], (230, 280))
Th_3d["T_w"] = avg(Th_3d[f"T_{MLD}"], (180, 230))
Th_3d["h"] = avg(Th_3d["thermocline"], (120, 280))
Th_3d["h_w"] = avg(Th_3d["thermocline"], (120, 180))
Th_3d["dTdx_total"] = get_ddx(Th_3d["T_50"].expand_dims({"latitude": [0]}))

Th_3d["Q_3"] = Th["Q_3"]
Th_3d["Q_34"] = Th["Q_34"]
Th_3d = Th_3d.drop_vars(["thermocline", "T_50", "T_80"])

### preprocess

In [ ]:
## standardize (for convenience)
Th /= Th.std()
Th_3d /= Th_3d.std()

## update thermocline variable
# Th["T_3"] = Th_3d["T_3"]
# Th["T_e"] = Th_3d["T_e"]
Th["h_mg"] = Th_3d["h"]
Th["h_w_mg"] = Th_3d["h_w"]

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)
# Th_rolling = src.utils.get_windowed(Th_3d, window_size=480, stride=120)

## drop last year because of NaNs
Th_rolling = Th_rolling.sel(year=slice(None, 2080))

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

#### eli

In [ ]:
# compute climatological gradient
dTdx_clim = Th_rolling["dTdx_total"].groupby("time.month").mean(["time"])

## get scaling factor
dTdx_scale = dTdx_clim / dTdx_clim.max()

## apply scaling
for v in list(Th_rolling):
    if "eli" in v:
        Th_rolling[f"{v}_scaled"] = Th_rolling[v].groupby("time.month") * dTdx_scale

        ## subtrack median
        grouped = Th_rolling[f"{v}_scaled"].groupby("time.month")
        Th_rolling[f"{v}_scaled_median"] = grouped - grouped.median(["member", "time"])

#### Remove linear dependence

Note: power point results use ```VARR=T_m```

In [ ]:
## use ML temp.
# Th_rolling["T_3"] = src.utils.get_windowed(
#     Th_3d["T_3"], window_size=480, stride=120,
# ).sel(year=slice(None,2080))

## specify var to remove
VARR = "T_3"

## get T^2
Th_rolling[f"{VARR}**2"] = Th_rolling[f"{VARR}"] ** 2

## remove SST dependence
for h_var in tqdm.tqdm(["h", "h_w", "h_ohc", "h_w_ohc", "h_mg", "h_w_mg"]):
    Th_rolling[f"{h_var}_hat"] = src.utils.remove_sst_dependence_v2(
        Th=Th_rolling,
        h_var=h_var,
        T_var=f"{VARR}",
        dims=["time", "member", "year"],
    )

    # Th_rolling[f"{h_var}_hat"] = src.utils.remove_sst_dependence_v2(
    #         Th=Th_rolling, h_var=f"{h_var}_hat", T_var=f"{VARR}**2",
    #     )

#### remove median

In [ ]:
## remove median (optional)
if REMOVE_MEDIAN:
    median = Th_rolling.groupby("time.month").median(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - median

else:
    mean = Th_rolling.groupby("time.month").mean(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - mean

## add constant
Th_rolling["ones"] = xr.ones_like(Th_rolling["T_3"])

## Compute time-varying RO parameters

### Fit model

Filenames:
- "T_3-h_w_ohc-zero_median_all-members.nc": ```maskb=["h"], maskNT=["T2", "T3", "TH"], maskNH=["T2"]```
- "T_3-h_w_ohc-zero_median.nc": same, but separate fits for each ensemble member
- "T_3-h_w_ohc-zero_median_simple.nc": ```maskNT=["T2", "T3"]```

In [ ]:
## gives reasonable results... (skewness too high for T though)
# varnames, Q_VAR = ["T_3", "h_mg"], "Q_3"

## gives reasonable results... (skewness too high for T though)
# varnames, Q_VAR = ["T_3", "h_ohc_hat"], "Q_3"

##
varnames, Q_VAR = ["T_3", "h_hat"], "Q_3"
# varnames, Q_VAR = ["T_e", "h_hat"], "Q_e"
# varnames, Q_VAR = ["T_3", "h_w"], "Q_3"
# varnames, Q_VAR = ["T_m", "h_hat"], "Q_m"

In [ ]:
## parameters for fitting
MODEL = src.XRO.XRO(
    ncycle=12, ac_order=3, taus=np.zeros(4, dtype=int), is_forward=False
)
# MODEL = src.XRO.XRO(ncycle=12, ac_order=3, is_forward=False)

## nonlinear R
fit_kwargs = dict(
    ac_mask_idx=None,
    # maskNT=["T2", "T3", "H2"],
    # maskNT=["T2", "T3"],
    # maskNT=["T2", "T3", "H2"],
    # maskNH=["T2"],
)

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=False,
    fname=None,
    # fname=f"{varnames[0]}-{varnames[1]}-zero_median_simple.nc",
    **fit_kwargs,
)

# ## get annual mean
# fits = MODEL.set_NRO_annualmean(fits)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

## compute error
fits["error"] = fits["Yfit"] - fits["Y"]
fits["error_amp"] = np.sqrt(fits["error"] ** 2)

## get corrected coords
time_coord = Th_rolling.stack(s=["member", "time"]).s
fits = fits.rename({"time": "s"}).assign_coords({"s": time_coord}).unstack("s")

## look at noise

In [ ]:
# delta =

xi = fits["xi_stdac"].isel(ranky=0)
R = fits["Lac"].isel(ranky=0, rankx=0) - fits["Lac"].isel(ranky=1, rankx=1)
F1 = fits["Lac"].isel(ranky=0, rankx=1)
F2 = -fits["Lac"].isel(ranky=1, rankx=0)
# beta= fits["NROT_Lac"].isel(nro_form=-1)
beta = fits["NROH_Lac"].isel(nro_form=0)
dxi = xi / xi.isel(year=0) - 1
dR = R / R.isel(year=0) - 1

fig, axs = plt.subplots(1, 4, figsize=(10, 2.5))
for ax, p in zip(axs, [R, xi, F1, beta]):

    for y in [1870, 1910, 1950, 1990]:

        ax.plot(params.cycle, p.sel(year=y))

axs[1].set_ylim([0, None])
axs[2].set_ylim([0, None])
src.utils.add_vticks(axs, xlines=[4, 6], xticks=[4, 6])

In [ ]:
BJ = fits["Lac"].isel(rankx=0, ranky=0) + fits["Lac"].isel(rankx=1, ranky=1)

fig, axs = plt.subplots(10, 1, figsize=(3, 11), layout="constrained")
for y in [1880, 1990]:  # , 2030, 2080]:
    axs[0].plot(fits.cycle, fits["Lac"].isel(rankx=0, ranky=0).sel(year=y), label=y)
    axs[1].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[2].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=2).sel(year=y), label=y)
    axs[3].plot(fits.cycle, fits["Lac"].isel(ranky=0, rankx=1).sel(year=y), label=y)
    axs[4].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=-1).sel(year=y), label=y)
    axs[5].plot(fits.cycle, fits["xi_stdac"].isel(ranky=0).sel(year=y), label=y)
    axs[6].plot(fits.cycle, -fits["Lac"].isel(ranky=1, rankx=0).sel(year=y), label=y)
    axs[7].plot(fits.cycle, -fits["NROH_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[8].plot(fits.cycle, -fits["Lac"].isel(rankx=1, ranky=1).sel(year=y), label=y)
    # axs[9].plot(fits.cycle, -fits["NLb_Lac"].isel(ranky=1).sel(year=y), label=y)
    axs[9].plot(fits.cycle, BJ.sel(year=y), label=y)


src.utils.add_vticks(axs, xlines=fits.cycle.values[[3, 5]], xticks=[])
kwargs = dict(x=1.0, y=0.5)
axs[0].text(s=r"$R$", transform=axs[0].transAxes, **kwargs)
axs[1].text(s=r"$\frac{dT}{dt}\propto T^2$", transform=axs[1].transAxes, **kwargs)
axs[2].text(s=r"$\frac{dT}{dt}\propto T^3$", transform=axs[2].transAxes, **kwargs)
axs[3].text(s=r"$F_1$", transform=axs[3].transAxes, **kwargs)
axs[4].text(s=r"$\frac{dT}{dt}\propto h^2$", transform=axs[4].transAxes, **kwargs)
axs[5].text(s=r"$\xi_T$", transform=axs[5].transAxes, **kwargs)
axs[6].text(s=r"$F_2$", transform=axs[6].transAxes, **kwargs)
axs[7].text(s=r"$\frac{dh}{dt}\propto -T^2$", transform=axs[7].transAxes, **kwargs)
axs[8].text(s=r"$\varepsilon$", transform=axs[8].transAxes, **kwargs)
# axs[9].text(s=r"$\frac{dh}{dt}\propto -h^2$", transform=axs[9].transAxes, **kwargs)
axs[9].text(s=r"$R-\varepsilon$", transform=axs[9].transAxes, **kwargs)
for ax in axs:
    ax.axhline(0, ls="--", c="gray", lw=0.8)
# ax.legend()

axs[-1].set_xticks(fits.cycle.values[[3, 5]], labels=[4, 6])
plt.show()

#### Hovmoller plot of $R$ and $\varepsilon$

In [ ]:
R_ = fits["Lac"].isel(rankx=0, ranky=0)
F1_ = fits["Lac"].isel(rankx=1, ranky=0)
R2 = fits["NROT_Lac"].isel(nro_form=0)
R3 = fits["NROT_Lac"].isel(nro_form=2)
eps_ = fits["Lac"].isel(rankx=1, ranky=1)
error_sigma = fits["xi_stdac"].isel(ranky=0)
# error_bar = fits["error"].groupby("time.month").mean(["time","member"]).isel(ranky=0)
# error_sigma =  fits["error"].groupby("time.month").std(["time","member"]).isel(ranky=0).rename({"month":"cycle"}).assign_coords({"cycle":R3.cycle})
# error_bar = (error_bar/error_sigma).rename({"month":"cycle"}).assign_coords({"cycle":R3.cycle})

In [ ]:
delta = lambda x: x - x.isel(year=0)
fig, axs = plt.subplots(1, 6, figsize=(12, 2), layout="constrained")

for ax, p, l in zip(
    axs,
    [R_, R_ + eps_, F1, R3, R2, error_sigma],
    [
        r"$R$",
        r"$R-\varepsilon$",
        r"$F_1$",
        r"$\frac{dT}{dt}\propto T^3$",
        r"$\frac{dT}{dt}\propto T^2$",
        "error",
    ],
):

    ax.contourf(
        p.cycle,
        p.year,
        delta(p),
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(0.75, 0.075),
        extend="both",
    )
    ax.axhline(1990, ls="--", c="gray", lw=1)
    ax.set_yticks([])
    ax.set_title(l)

src.utils.add_vticks(
    axs,
    xticks=[],
    xlines=[p.cycle[i] for i in [0, 3, 5]],
)


axs[0].set_yticks([1870, 1990, 2080])
plt.show()

In [ ]:
## check fits make sense
idx = dict(year=2, member=0, time=slice(None, 120))

## make plot
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(fits["Y"].isel(ranky=0, **idx))
ax.plot(get_ddt(Th_rolling[varnames[0]].isel(**idx)), ls="--")
plt.show()

In [ ]:
## resample to AMJ
fit_amj = fits.resample({"time": "QS-JAN"}).mean().isel(time=slice(1, -4, 4))
fit_a = fits.isel(time=slice(3, -12, 12))
fit_amj0 = fit_amj.sel(year=1880)
fit_amj1 = fit_amj.sel(year=1990)
fit_a0 = fit_a.sel(year=1880)
fit_a1 = fit_a.sel(year=1990)

In [ ]:
for fit_amj_, fit_a_ in zip([fit_amj0, fit_amj1], [fit_a0, fit_a1]):

    fig, axs = plt.subplots(2, 2, figsize=(5, 5), layout="constrained")

    ## loop thru (actual, error)
    for j, v in enumerate(["Y", "error"]):

        for xi, ax in enumerate(axs[j]):

            x0 = fit_a_["X"].isel(rankx=xi)
            # x1 = fit_a_[v].isel(ranky=0).transpose(*x0.dims)
            # x0 = fit_amj_["X"].isel(rankx=xi)
            x1 = fit_amj_[v].isel(ranky=0).transpose(*x0.dims)

            ax.scatter(x0, x1, s=3, alpha=0.5)
            corr = xr.corr(x0, x1).values.item()
            ax.set_title(f"Corr: {corr:.2f}")

            ## format
            ax_kwargs = dict(ls="--", c="k", lw=0.8)
            ax.axhline(0, **ax_kwargs)
            ax.axvline(0, **ax_kwargs)

        src.utils.set_lims(axs[j])
        axs[j, 1].set_yticks([])

    for ax in axs[0]:
        ax.set_xticks([])

    plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

for ax, year in zip(axs, [1880, 1990]):
    ax.scatter(
        fit_amj["Y"].isel(ranky=0).sel(year=year),
        fit_amj["Yfit"].isel(ranky=0).sel(year=year),
        s=3,
        alpha=0.5,
    )

    ## format
    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    zz = np.linspace(-10, 5)
    ax.plot(zz, zz, ls="-", lw=0.8, c="k")

axs[1].set_yticks([])
src.utils.set_lims(axs)

plt.show()

Correlation with SST

In [ ]:
# anom["ddt_T"] = get_ddt(anom["T"])
anom["ddt_T"] = get_ddt(anom["sst"])

In [ ]:
## get SST data to match
anom_ = src.utils.get_windowed(
    anom[["T", "sst", "ssh", "w", "ddt_T", "taux"]], window_size=480, stride=120
).compute()

print("Here")
anom_ = anom_.isel(year=slice(None, -1))
anom_ = xr.merge([anom_, Th_rolling])

## resample to AMJ
anom_amj = anom_.resample({"time": "QS-JAN"}).mean().isel(time=slice(1, None, 4))
anom_jfm = anom_.resample({"time": "QS-JAN"}).mean().isel(time=slice(0, None, 4))
anom_jfm = anom_jfm.assign_coords({"time": anom_amj.time})

print("Here")
## add error to spatial arrays
anom_amj["error"] = fit_amj["error"].isel(ranky=0)
anom_amj["error_amp"] = fit_amj["error_amp"].isel(ranky=0)
anom_jfm["error"] = fit_amj["error"].isel(ranky=0)
anom_jfm["error_amp"] = fit_amj["error_amp"].isel(ranky=0)

## compute SPMM
anom_amj["spmm"] = src.utils.reconstruct_fn(
    scores=anom_amj["sst"],
    components=anom["sst_comp"],
    fn=lambda x: x.sel(latitude=slice(-15, -5), longitude=slice(210, 270)).mean(
        ["latitude", "longitude"]
    ),
)

## check it mathces
print(f"{xr.corr(anom_amj["T_3"], fit_amj["X"].isel(rankx=0)).values.item():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(anom_amj["T_3"].isel(idx))
ax.plot(fit_amj["X"].isel(rankx=0).isel(idx))
plt.show()

In [ ]:
anom_amj["w_3"] = src.utils.reconstruct_fn(
    scores=anom_amj["w"],
    components=forced["w_comp"],
    fn=lambda x: x.sel(longitude=slice(210, 280), z_t=slice(30, 80)).mean(
        ["z_t", "longitude"]
    ),
)

anom_amj["ddt_T_3"] = src.utils.reconstruct_fn(
    scores=anom_amj["ddt_T"],
    components=forced["sst_comp"],
    fn=src.utils.get_nino3,
    # components=forced["T_comp"],
    # fn=lambda x : x.sel(
    #     longitude=slice(210,270), z_t=slice(0,80)
    # ).mean(["z_t","longitude"]),
)

anom_amj["taux_3"] = src.utils.reconstruct_fn(
    scores=anom_amj["taux"],
    components=forced["taux_comp"],
    fn=lambda x: x.sel(longitude=slice(180, 270), latitude=slice(-5, 5)).mean(
        ["latitude", "longitude"]
    ),
)

anom_amj["h_e"] = src.utils.reconstruct_fn(
    scores=anom_amj["ssh"],
    components=forced["ssh_comp"],
    fn=lambda x: x.sel(longitude=slice(210, 270), latitude=slice(-2, 2)).mean(
        ["latitude", "longitude"]
    ),
)

# anom_amj["pr_3"] = src.utils.reconstruct_fn(
#     scores=anom_amj["pr"],
#     components=forced["pr_comp"],
#     fn=lambda x : x.sel(
#         longitude=slice(210,270), latitude=slice(-5,5)
#     ).mean(["latitude","longitude"]),
# )

anom_amj["dTdz_3"] = src.utils.reconstruct_fn(
    scores=anom_amj["T"],
    components=forced["T_comp"],
    fn=lambda x: x.sel(z_t=50, method="nearest")
    .sel(longitude=slice(240, 270))
    .mean(["longitude"]),
)

In [ ]:
# forced_amj =
forced_amj = (
    forced.isel(time=slice(None, 480))
    .groupby("time.month")
    .mean()
    .sel(month=slice(4, 6))
    .mean("month")
)
dTdz_clim = src.utils.reconstruct_fn(
    scores=forced_amj["T"],
    components=forced["T_comp"],
    fn=lambda x: x.differentiate("z_t"),
)

vol_avg = lambda x: x.sel(z_t=slice(0, 80), longitude=slice(210, 280)).mean(
    ["z_t", "longitude"]
)
anom_amj["ekf"] = (vol_avg(dTdz_clim * forced["w_comp"]) * anom_amj["w"]).sum("mode")

In [ ]:
d = lambda x: x / x.isel(year=0) - 1
# plt.plot(anom_amj.year, d(anom_amj["w_3"].std(["time","member"])))
# plt.plot(anom_amj.year, d(anom_amj["ekf"].std(["time","member"])))
plt.plot(anom_amj.year, d(anom_amj["taux_3"].std(["time", "member"])))
plt.plot(anom_amj.year, d(anom_amj["T_3"].std(["time", "member"])))
# plt.plot(anom_amj.year, d(anom_amj["T_3"].std(["time","member"])))
# plt.plot(anom_amj.year, d(anom_amj["pr_3"].std(["time","member"])))
# plt.plot(anom_amj.year, d(anom_amj["error"].std(["time","member"])))
# plt.ylim([-.02,.1])
# plt.xlim([1870, 2000])

In [ ]:
bins = np.linspace(-7, 7, 20)


fig, ax = plt.subplots(figsize=(6, 3))
for y in [1870, 1990]:

    ## get error
    # e = anom_amj["error"].sel(year=y).values.flatten()
    # e = fit_a["error"].isel(ranky=0).sel(year=y).values.flatten()
    e = fits["error"].isel(time=slice(5, -12, 12), ranky=0).sel(year=y).values.flatten()
    e = e[~np.isnan(e)]

    ## pdf
    pdf, _ = src.utils.get_empirical_pdf(e, edges=bins)

    # stats
    print(f"mu:     {e.mean():.2f}")
    print(f"median: {np.median(e):.2f}")
    print(f"sigma:  {e.std():.2f}")
    print(f"skew:   {scipy.stats.skew(e):.2f}")
    print()

    ax.stairs(pdf, bins)

ax.axvline(0, ls="--", c="k")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
for j in np.arange(4, 7):
    e = fits["error"].isel(time=slice(j, -12, 12), ranky=0)
    ax.plot(e.year, e.mean(["member", "time"]) / e.std(["member", "time"]))
ax.axhline(0, ls="--", c="k")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

for ax, year in zip(axs, [1870, 1990]):

    x0 = anom_amj["ekf"].sel(year=year)
    # x0 = anom_amj["dTdz_3"].sel(year=year)
    # x1 = anom_amj["ddt_T_3"].sel(year=year)#.isel(time=slice(None,-1))
    # x0 = anom_amj["taux_3"].sel(year=year)
    # x0 = anom_amj["Q_3"].sel(year=year)
    # x1 = anom_amj["w_3"].sel(year=year)
    # x1 = anom_amj["h_w_ohc"].sel(year=year)
    # x0 = anom_amj["T_w"].sel(year=year)
    # x0 = anom_amj["h_e"].sel(year=year)
    x0 = (anom_amj["h_hat"]).sel(year=year)
    # x0 = fit_a["X"].isel(rankx=0).sel(year=year)
    # x0 = anom_amj["eli_05"].sel(year=year)
    x1 = anom_amj["error"].sel(year=year)

    ax.scatter(
        x0,
        x1.transpose(*x0.dims),
        # anom_jfm["T_34"].sel(year=year),
        # anom_amj[varnames[0]].sel(year=year),
        # anom_amj["error_amp"].sel(year=year),
        # anom_amj["error"].sel(year=year),
        s=3,
        alpha=0.5,
    )

    ## format
    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    slope = xr.cov(x0, x1) / x0.var()
    corr0 = slope * x0.std() / x1.std()
    corr = xr.corr(x0, x1)
    ax.set_title(f"Corr = {corr:.2f}, $m=$ {slope:.1e}")

    ## get reconstruction
    x1_recon = slope * x0
    print(f"{(x1-x1_recon).std().values.item():.2f}")
    print(f"{(x1).std().values.item():.2f}")
    print()
    # zz=np.linspace(-10,5)
    # ax.plot(zz,zz,ls="-", lw=.8, c="k")

axs[1].set_yticks([])
src.utils.set_lims(axs)

plt.show()

In [ ]:
for a_ in [anom_jfm, anom_amj]:
    print(
        xr.corr(
            a_[["T_4", "T_34", "T_3", "h", "h_w", varnames[1]]].to_dataarray(),
            a_["error_amp"],
            # a_["error"],
            dim=["time", "member"],
        )
        .isel(year=1)
        .values
    )
    print()

In [ ]:
is_pos = anom_jfm[varnames[0]] >= 1
is_neg = anom_jfm[varnames[0]] <= 1
error_amp_pos = anom_amj["error_amp"].where(is_pos).mean(["time"])
error_amp_neg = anom_amj["error_amp"].where(is_neg).mean(["time"])

In [ ]:
# delta = lambda x: x / x.isel(year=1) - 1
delta = lambda x: x
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(anom_amj.year, delta(error_amp_neg.mean("member")))
ax.plot(anom_amj.year, delta(error_amp_pos.mean("member")))
ax.plot(anom_amj.year, delta(anom_amj["error_amp"].mean(["time", "member"])), c="k")
ax.set_ylim([0, None])
plt.show()

In [ ]:
# x0 = anom_jfm["T_3"].std(["time"])
# x0 = anom_amj["T_3"].std(["time"])
x0 = fit_a["X"].isel(rankx=0).std(["time"])
# x1 = anom_amj["error"].std("time")
x1 = anom_amj["error_amp"].mean("time")

fig, ax = plt.subplots(figsize=(4, 3), layout="constrained")

for y, m in zip([1870, 2010], ["o", "x"]):
    x0_ = x0.sel(year=y)
    x1_ = x1.sel(year=y)
    corr = xr.corr(x0_, x1_).values.item()
    # ax.set_title(f"corr = {xr.corr(x0_,x1_).values.item():.2f}")

    ax.scatter(x0_, x1_, marker=m, label=f"corr = {corr:.2f}")
ax.legend()
# src.utils.set_lims(axs)
plt.show()

#### Regress on error

In [ ]:
## regress on error
m_error_ = src.utils.regress_xr_proj(
    # data=anom_jfm,
    # x_vars=["error_amp"],
    data=anom_amj,
    # x_vars=["error_amp"],
    x_vars=["error"],
    y_vars=["sst", "T", "w", "ddt_T"],
    # x_vars=["error"],
).squeeze(drop=True)

## get spatial patterns
m_error = src.utils.reconstruct_wrapper(
    xr.merge([m_error_, anom[["T_comp", "sst_comp", "w_comp"]]]),
    fn=lambda x: x,
)

# m_error["ddt_T"] = src.utils.reconstruct_fn(
#     components=anom["T_comp"], scores=m_error_["ddt_T"], fn=lambda x : x,
# )
m_error["ddt_T"] = src.utils.reconstruct_fn(
    components=anom["sst_comp"],
    scores=m_error_["ddt_T"],
    fn=lambda x: x,
)

#### Plot

In [ ]:
import cartopy.crs as ccrs

varname = "ddt_T"

if varname == "pr":
    amp = 2e-5
else:
    # amp = .3
    amp = 1.5

fig = plt.figure(figsize=(8, 4.5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=20)
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)

kwargs = dict(
    cmap="cmo.balance",
    transform=ccrs.PlateCarree(),
    # levels=src.utils.make_cb_range(0.3, 0.03),
    levels=src.utils.make_cb_range(amp, amp / 10),
    extend="both",
)

for j, y in enumerate([1880, 2000]):

    axs[j, 0].contourf(
        m_error.longitude,
        m_error.latitude,
        m_error[varname].sel(year=y),
        **kwargs,
    )

axs[-1, 0].contourf(
    m_error.longitude,
    m_error.latitude,
    3 * (m_error[varname].sel(year=2000) - m_error[varname].sel(year=1880)),
    **kwargs,
)


for ax in axs[:, 0]:
    src.utils.plot_nino3_box(ax, c="k", lw=1, ls="--", alpha=0.5)
    # src.utils.plot_box(
    #     ax, lats=[-5, 5], lons=[190, 240], c="k", lw=1, ls="--", alpha=0.5
    # )
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

kwargs = dict(
    cmap="cmo.balance", levels=src.utils.make_cb_range(0.4, 0.04), extend="both"
)

for ax, y in zip(axs, [1880, 2000]):

    ax.contourf(m_error.longitude, m_error.z_t, m_error["T"].sel(year=y), **kwargs)

axs[-1].contourf(
    m_error.longitude,
    m_error.z_t,
    6 * (m_error["T"].sel(year=2000) - m_error["T"].sel(year=1880)),
    **kwargs
)

for ax in axs:
    ax.set_ylim([300, 5])
    ax.set_xlim([130, 285])
    ax.set_yticks([])
    ax.axhline(75, ls="--", c="gray", lw=0.8)

axs[0].set_yticks([75, 150])
src.utils.add_vticks(axs, xlines=[210, 270], xticks=[130, 210, 270])
plt.show()

In [ ]:
# fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

# kwargs = dict(cmap="cmo.balance", levels=src.utils.make_cb_range(3, 0.3), extend="both")

# for ax, y in zip(axs, [1880, 1990]):

#     ax.contourf(m_error.longitude, m_error.z_t, m_error["ddt_T"].sel(year=y), **kwargs)

# axs[-1].contourf(
#     m_error.longitude,
#     m_error.z_t,
#     6 * (m_error["ddt_T"].sel(year=1990) - m_error["ddt_T"].sel(year=1880)),
#     **kwargs
# )

# for ax in axs:
#     ax.set_ylim([150, 5])
#     ax.set_xlim([130, 285])
#     ax.set_yticks([])
#     ax.axhline(75, ls="--", c="gray", lw=0.8)

# axs[0].set_yticks([75, 150])
# src.utils.add_vticks(axs, xlines=[210, 270], xticks=[130, 210, 270])
# plt.show()

In [ ]:
# corrs = []
# for y in anom_amj.year:

#     ## get standard dev of w
#     w_sigma = src.utils.reconstruct_std(
#         scores=anom_amj["w"].sel(year=y).fillna(0.0),
#         components=forced["w_comp"],
#     )

#     ## get standard dev of error
#     e_sigma = anom_amj["error"].sel(year=y).std()

#     ## compute correlation
#     corrs.append( m_error["w"].sel(year=y) * e_sigma / w_sigma)

# corrs = xr.concat(corrs, dim=anom_amj.year)

In [ ]:
# fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

# kwargs = dict(
#     cmap="cmo.balance", levels=src.utils.make_cb_range(.5, .05), extend="both"
# )

# for ax, y in zip(axs, [1880, 2000]):

#     ax.contourf(m_error.longitude, m_error.z_t, corrs.sel(year=y), **kwargs)

# axs[-1].contourf(
#     m_error.longitude,
#     m_error.z_t,
#     6 * (corrs.sel(year=2000) - corrs.sel(year=1880)),
#     **kwargs
# )

# for ax in axs:
#     ax.set_ylim([150, 5])
#     ax.set_xlim([130, 285])
#     ax.set_yticks([])
#     ax.axhline(75, ls="--", c="gray", lw=0.8)

# axs[0].set_yticks([75, 150])
# src.utils.add_vticks(axs, xlines=[210, 270], xticks=[130, 210, 270])
# plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

kwargs = dict(cmap="cmo.balance", levels=src.utils.make_cb_range(3, 0.3), extend="both")

for ax, y in zip(axs, [1880, 2000]):

    ax.contourf(m_error.longitude, m_error.z_t, m_error["w"].sel(year=y), **kwargs)

axs[-1].contourf(
    m_error.longitude,
    m_error.z_t,
    6 * (m_error["w"].sel(year=2000) - m_error["w"].sel(year=1880)),
    **kwargs
)

for ax in axs:
    ax.set_ylim([150, 5])
    ax.set_xlim([130, 285])
    ax.set_yticks([])
    ax.axhline(75, ls="--", c="gray", lw=0.8)

axs[0].set_yticks([75, 150])
src.utils.add_vticks(axs, xlines=[210, 270], xticks=[130, 210, 270])
plt.show()